In [3]:
import pandas as pd
import numpy as np
import re
from ast import literal_eval
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from defenders.pii_detection.hmm.src.HMM import HMM

from defenders.pii_detection.hmm.src.word_features import extract_word_features , NER_TAG_TO_INDEX, INDEX_TO_NER_TAG ,NER_TAGS



In [4]:
df = pd.read_parquet('../../data/word_based_data.parquet')

train_df, test_df = train_test_split(df, test_size=0.1, random_state=42)
train_df, val_df  = train_test_split(train_df, test_size=0.1, random_state=42)
print(len(train_df), len(val_df), len(test_df))


17486 1943 2159


In [5]:
def encode_tags(tag_seqs):
    return [[NER_TAG_TO_INDEX.get(t, NER_TAG_TO_INDEX['O']) for t in seq] for seq in tag_seqs]

def encode_words_baseline(word_seqs, vocab):
    return [[vocab.get(w, vocab['UNK']) for w in seq] for seq in word_seqs]


In [6]:

train_words_raw = [list(x) for x in train_df['words'].tolist()]
train_tags_raw  = [list(x) for x in train_df['labels'].tolist()]




In [7]:

baseline_vocab = {}
for seq in train_words_raw:
    for w in seq:
        if w not in baseline_vocab:
            baseline_vocab[w] = len(baseline_vocab)
baseline_vocab['UNK'] = len(baseline_vocab)

In [8]:

train_tags  = encode_tags(train_tags_raw)
train_obs   = encode_words_baseline(train_words_raw, baseline_vocab)


In [9]:

test_words_raw = [list(x) for x in test_df['words'].tolist()]
test_tags_raw  = [list(x) for x in test_df['labels'].tolist()]

test_tags = encode_tags(test_tags_raw)
test_obs   = encode_words_baseline(test_words_raw, baseline_vocab)

In [10]:
baseline_model = HMM(num_states=len(NER_TAGS), vocab_size=len(baseline_vocab))
baseline_model.train(train_tags, train_obs)
print("Baseline training complete.")


Baseline training complete.


In [11]:

baseline_preds = [baseline_model.predict(seq) for seq in test_obs]

baseline_acc = np.mean([np.array_equal(p, t) for p, t in zip(baseline_preds, test_tags)])
#flatten
true_flat = [t for seq in test_tags  for t in seq]
pred_flat= [t for seq in baseline_preds for t in seq]

print("\nBaseline Classification Report:")
print(classification_report(
    true_flat, pred_flat,
    labels=list(range(len(NER_TAGS))),
    target_names=NER_TAGS,
    zero_division=0
))



Baseline Classification Report:
                    precision    recall  f1-score   support

     B-ACCOUNTNAME       1.00      0.19      0.33       103
   B-ACCOUNTNUMBER       0.00      0.00      0.00       104
B-CREDITCARDNUMBER       0.00      0.00      0.00       105
           B-EMAIL       0.00      0.00      0.00       137
              B-IP       0.00      0.00      0.00        75
            B-IPV4       0.00      0.00      0.00       118
            B-IPV6       0.00      0.00      0.00       103
             B-MAC       0.00      0.00      0.00        79
        B-PASSWORD       0.00      0.00      0.00       109
    B-PHONE_NUMBER       0.00      0.00      0.00       117
             B-SSN       0.00      0.00      0.00        79
        B-USERNAME       0.00      0.00      0.00       139
                 O       0.98      1.00      0.99     65421

          accuracy                           0.98     66689
         macro avg       0.15      0.09      0.10     66689
     

In [12]:
from seqeval.metrics import classification_report as seqeval_report
from seqeval.metrics import f1_score
print("\nSeqeval Classification Report:")
print(seqeval_report(
    [[INDEX_TO_NER_TAG[t] for t in seq] for seq in test_tags],
    [[INDEX_TO_NER_TAG[t] for t in seq] for seq in baseline_preds],
    zero_division=0
))
print("Span-Level F1:", f1_score(
    [[INDEX_TO_NER_TAG[t] for t in seq] for seq in test_tags],
    [[INDEX_TO_NER_TAG[t] for t in seq] for seq in baseline_preds],
    average='micro'
))


Seqeval Classification Report:
                  precision    recall  f1-score   support

     ACCOUNTNAME       1.00      0.19      0.33       103
   ACCOUNTNUMBER       0.00      0.00      0.00       104
CREDITCARDNUMBER       0.00      0.00      0.00       105
           EMAIL       0.00      0.00      0.00       137
              IP       0.00      0.00      0.00        75
            IPV4       0.00      0.00      0.00       118
            IPV6       0.00      0.00      0.00       103
             MAC       0.00      0.00      0.00        79
        PASSWORD       0.00      0.00      0.00       109
    PHONE_NUMBER       0.00      0.00      0.00       117
             SSN       0.00      0.00      0.00        79
        USERNAME       0.00      0.00      0.00       139

       micro avg       1.00      0.02      0.03      1268
       macro avg       0.08      0.02      0.03      1268
    weighted avg       0.08      0.02      0.03      1268

Span-Level F1: 0.031055900621118012


In [13]:
# build new vocabulary 
enhanced_vocab = {}

def register(tok):
    if tok not in enhanced_vocab:
        enhanced_vocab[tok] = len(enhanced_vocab)

for seq in train_words_raw:
    for w in seq:
        register(w)
        for f in extract_word_features(w):
            register(f)

enhanced_vocab['UNK'] = len(enhanced_vocab)


In [14]:
def word_to_feature_indices(word, vocab):
    tokens = [word] + extract_word_features(word)
    return [vocab.get(t, vocab['UNK']) for t in tokens]

def encode_words_enhanced(word_seqs, vocab):
    return [[word_to_feature_indices(w, vocab) for w in seq] for seq in word_seqs]


In [15]:

train_obs_enhanced = encode_words_enhanced(train_words_raw, enhanced_vocab)
test_obs_enhanced  = encode_words_enhanced(test_words_raw,  enhanced_vocab)

train_tags_enhanced = train_tags
test_tags_enhanced  = test_tags

print(f"Enhanced vocab size: {len(enhanced_vocab)}")


Enhanced vocab size: 138676


In [16]:
#train
enhanced_model = HMM(num_states=len(NER_TAGS), vocab_size=len(enhanced_vocab))
enhanced_model.train(train_tags_enhanced, train_obs_enhanced)
print("Enhanced training complete.")


Enhanced training complete.


In [17]:

enhanced_preds = [enhanced_model.predict(seq) for seq in test_obs_enhanced]

enhanced_acc = np.mean([np.array_equal(p, t) for p, t in zip(enhanced_preds, test_tags_enhanced)])
# flatten
true_flat_e = [t for seq in test_tags_enhanced    for t in seq]
pred_flat_e = [t for seq in enhanced_preds for t in seq]

print("\nEnhanced Classification Report:")
print(classification_report(
    true_flat_e, pred_flat_e,
    labels=list(range(len(NER_TAGS))),
    target_names=NER_TAGS,
    zero_division=0
))



Enhanced Classification Report:
                    precision    recall  f1-score   support

     B-ACCOUNTNAME       0.91      0.51      0.66       103
   B-ACCOUNTNUMBER       0.16      0.83      0.26       104
B-CREDITCARDNUMBER       0.30      0.66      0.41       105
           B-EMAIL       0.78      0.99      0.87       137
              B-IP       0.44      0.11      0.17        75
            B-IPV4       0.37      0.97      0.54       118
            B-IPV6       0.54      0.66      0.59       103
             B-MAC       0.70      0.73      0.72        79
        B-PASSWORD       0.31      0.72      0.43       109
    B-PHONE_NUMBER       0.22      0.64      0.33       117
             B-SSN       0.62      0.73      0.67        79
        B-USERNAME       0.41      0.39      0.40       139
                 O       1.00      0.98      0.99     65421

          accuracy                           0.98     66689
         macro avg       0.52      0.69      0.54     66689
     

In [18]:
from seqeval.metrics import classification_report as seqeval_report
from seqeval.metrics import f1_score
print("\nSeqeval Classification Report:")
print(seqeval_report(
    [[INDEX_TO_NER_TAG[t] for t in seq] for seq in test_tags_enhanced],
    [[INDEX_TO_NER_TAG[t] for t in seq] for seq in enhanced_preds],
    zero_division=0
))
print("Span-Level F1:", f1_score(
    [[INDEX_TO_NER_TAG[t] for t in seq] for seq in test_tags_enhanced],
    [[INDEX_TO_NER_TAG[t] for t in seq] for seq in enhanced_preds],

    average='micro'
))


Seqeval Classification Report:
                  precision    recall  f1-score   support

     ACCOUNTNAME       0.91      0.51      0.66       103
   ACCOUNTNUMBER       0.16      0.83      0.26       104
CREDITCARDNUMBER       0.30      0.66      0.41       105
           EMAIL       0.78      0.99      0.87       137
              IP       0.44      0.11      0.17        75
            IPV4       0.37      0.97      0.54       118
            IPV6       0.54      0.66      0.59       103
             MAC       0.70      0.73      0.72        79
        PASSWORD       0.31      0.72      0.43       109
    PHONE_NUMBER       0.22      0.64      0.33       117
             SSN       0.62      0.73      0.67        79
        USERNAME       0.41      0.39      0.40       139

       micro avg       0.36      0.68      0.47      1268
       macro avg       0.48      0.66      0.50      1268
    weighted avg       0.47      0.68      0.51      1268

Span-Level F1: 0.4724669603524229
